# 01 Broad-Board Ablation

This notebook is the first evidence notebook in the public release suite.

## Purpose

It reproduces the **broad-board resolution ablation** across the continuous **16.5B to 31.5B** seam-stitched board and shows that:

- the dominant organizing coordinate remains **`h1`**
- the trimmed late-stage continuation winner remains **quadratic** for both `b1` and `b2`
- the broad-board evidence is stable across **130k, 90k, 70k, and 52k** row sizes

This notebook is designed to be **artifact-first**. By default it loads frozen summary JSON files and rebuilds the headline table and plot without rerunning the heavy extraction pipeline.


## Reading note

This notebook supports the broad-board claims in the paper. It is intentionally narrower than the full working board.

For the rest of the public evidence chain, continue with:

- `02_terminal_ladder.ipynb`
- `03_terminal_8k_matched_packet_null.ipynb`
- `04_terminal_8k_jackknife.ipynb`


In [ ]:
# Optional path settings for Colab or local runs

import os
import json
from pathlib import Path

DEFAULT_OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/ForgeV16c"
OUT_DIR = os.environ.get("FORGE_V16C_OUT_DIR", DEFAULT_OUT_DIR)

print("OUT_DIR =", OUT_DIR)

## Expected frozen artifacts

This notebook looks for the following summary files:

- `q1_bigband_continuity_summary_v16c_130k_31p5B.json`
- `q1_bigband_continuity_summary_v16c_90k_31p5B.json`
- `q1_bigband_continuity_summary_v16c_70k_31p5B.json`
- `q1_bigband_continuity_summary_v16c_52k_31p5B.json`

and:

- `r1_trimmed_latec_summary_v16c_130k_31p5B.json`
- `r1_trimmed_latec_summary_v16c_90k_31p5B.json`
- `r1_trimmed_latec_summary_v16c_70k_31p5B.json`
- `r1_trimmed_latec_summary_v16c_52k_31p5B.json`

If your filenames differ slightly, edit the `FILES` map below.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FILES = {
    "130k": {
        "q1_summary": os.path.join(OUT_DIR, "q1_bigband_continuity_summary_v16c_130k_31p5B.json"),
        "r1_summary": os.path.join(OUT_DIR, "r1_trimmed_latec_summary_v16c_130k_31p5B.json"),
    },
    "90k": {
        "q1_summary": os.path.join(OUT_DIR, "q1_bigband_continuity_summary_v16c_90k_31p5B.json"),
        "r1_summary": os.path.join(OUT_DIR, "r1_trimmed_latec_summary_v16c_90k_31p5B.json"),
    },
    "70k": {
        "q1_summary": os.path.join(OUT_DIR, "q1_bigband_continuity_summary_v16c_70k_31p5B.json"),
        "r1_summary": os.path.join(OUT_DIR, "r1_trimmed_latec_summary_v16c_70k_31p5B.json"),
    },
    "52k": {
        "q1_summary": os.path.join(OUT_DIR, "q1_bigband_continuity_summary_v16c_52k_31p5B.json"),
        "r1_summary": os.path.join(OUT_DIR, "r1_trimmed_latec_summary_v16c_52k_31p5B.json"),
    },
}

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

for label, paths in FILES.items():
    print(f"--- {label} ---")
    for k, p in paths.items():
        print(f"{k:>10}: {'FOUND' if os.path.exists(p) else 'MISSING'}")
        print(f"          {p}")

## Load the broad-board summaries

This cell builds a compact comparison table from the frozen Q1 and trimmed late-stage summaries.


In [ ]:
rows = []

for label, paths in FILES.items():
    q1 = load_json(paths["q1_summary"])
    r1 = load_json(paths["r1_summary"])

    row = {
        "chunk_label": label,
        "rows_built": q1.get("rows_built"),
        "altitude_min_m": q1.get("altitude_min_m"),
        "altitude_max_m": q1.get("altitude_max_m"),
        "q1_rmse_pure": q1.get("rmse_pure"),
        "q1_rmse_fit": q1.get("rmse_fit"),
        "q1_boundary_cross_rows": q1.get("boundary_cross_rows"),
        "q1_near_edge_rows": q1.get("near_edge_rows"),
        "q1_pure_interior_rows": q1.get("pure_interior_rows"),
        "q1_fitted_a": q1.get("fitted_a"),
        "q1_fitted_b": q1.get("fitted_b"),
        "packet_count": r1.get("packet_count"),
        "eligible_row_count": r1.get("eligible_row_count"),
        "compression_threshold": r1.get("compression_threshold"),
        "best_candidate": r1.get("best_candidate"),
        "best_harmonic": r1.get("best_harmonic"),
        "best_resultant_r": r1.get("best_resultant_r"),
        "best_arc80": r1.get("best_arc80"),
        "rolling_windows": r1.get("rolling_windows"),
        "best_b1_model": r1.get("best_b1_model"),
        "best_b1_test_mae": r1.get("best_b1_test_mae"),
        "best_b2_model": r1.get("best_b2_model"),
        "best_b2_test_mae": r1.get("best_b2_test_mae"),
        "b1_residual_signs": str(r1.get("b1_residual_signs")),
        "b2_residual_signs": str(r1.get("b2_residual_signs")),
    }
    rows.append(row)

broad_df = pd.DataFrame(rows)
order = ["130k", "90k", "70k", "52k"]
broad_df["chunk_label"] = pd.Categorical(broad_df["chunk_label"], order, ordered=True)
broad_df = broad_df.sort_values("chunk_label").reset_index(drop=True)

display(broad_df)

## Headline interpretation

The broad-board ablation supports three claims:

1. **Best harmonic stability**  
   The dominant harmonic remains `h1` at every tested row size.

2. **Continuation-family stability**  
   The best model family for both `b1` and `b2` remains quadratic.

3. **Nontrivial row-size robustness**  
   The board is rebuilt at materially different fixed row sizes, yet the same organizing harmonic and continuation family persist.


In [ ]:
headline_cols = [
    "chunk_label",
    "rows_built",
    "best_candidate",
    "best_resultant_r",
    "best_arc80",
    "best_b1_model",
    "best_b2_model",
    "rolling_windows",
]
display(broad_df[headline_cols])

## Plot the broad-board stability

This plot is a compact visual summary of the broad-board ablation.


In [ ]:
plot_df = broad_df.copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

ax = axes[0]
ax.plot(plot_df["chunk_label"].astype(str), plot_df["best_resultant_r"], marker="o")
ax.set_title("A. Best resultant R across broad-board sizes")
ax.set_xlabel("Row size")
ax.set_ylabel("Resultant R")
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(plot_df["chunk_label"].astype(str), plot_df["best_arc80"], marker="o")
ax.set_title("B. Best Arc80 across broad-board sizes")
ax.set_xlabel("Row size")
ax.set_ylabel("Arc80")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Sanity checks

These are the simple release-level checks this notebook should satisfy:

- `best_candidate` is `logT_over_pi_h1` for all four row sizes
- `best_b1_model` is `poly_deg2` for all four row sizes
- `best_b2_model` is `poly_deg2` for all four row sizes
- `best_resultant_r` stays high and stable across the ladder


In [ ]:
checks = pd.DataFrame({
    "chunk_label": broad_df["chunk_label"].astype(str),
    "h1_winner": broad_df["best_candidate"].eq("logT_over_pi_h1"),
    "b1_poly_deg2": broad_df["best_b1_model"].eq("poly_deg2"),
    "b2_poly_deg2": broad_df["best_b2_model"].eq("poly_deg2"),
})

display(checks)

if checks[["h1_winner", "b1_poly_deg2", "b2_poly_deg2"]].all().all():
    print("All broad-board release checks passed.")
else:
    print("One or more broad-board release checks failed. Inspect the summary files.")

## Optional export

If you want to save the compact release table into the repository artifacts folder, run the next cell.


In [ ]:
# Optional export
# export_path = os.path.join(OUT_DIR, "release_broad_board_ablation_summary.csv")
# broad_df.to_csv(export_path, index=False)
# print("Saved:", export_path)

## Next notebook

Continue to:

**`02_terminal_ladder.ipynb`**

That notebook reproduces the 52k / 8k / 1k terminal comparator and shows that the terminal tightening is a band property rather than a 1k-only artifact.
